In [2]:
from pyspark.sql import functions as F

def write_gold(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print(name, spark.table(name).count())

silver = lambda t: spark.table(f"lh_silver.{t}")

# dim_date: full calendar years covering all orders
b = silver("dbo.orders").agg(F.min("order_date").alias("mn"), F.max("order_date").alias("mx")).first()
dim_date = (spark.sql(f"""SELECT explode(sequence(to_date('{b.mn.year}-01-01'),
                         to_date('{b.mx.year}-12-31'), interval 1 day)) AS date""")
    .select(F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"), "date",
            F.year("date").alias("year"), F.quarter("date").alias("quarter"),
            F.month("date").alias("month_no"), F.date_format("date", "MMM").alias("month_name"),
            F.date_format("date", "yyyy-MM").alias("year_month"),
            F.weekofyear("date").alias("week_of_year"), F.date_format("date", "E").alias("day_name"),
            F.dayofweek("date").isin(1, 7).alias("is_weekend")))
write_gold(dim_date, "dim_date")

dim_customer = silver("dbo.customers").select(
    F.xxhash64("customer_id").alias("customer_key"), "customer_id", "customer_name",
    "segment", "city", "state", "postal_code", "country")
write_gold(dim_customer, "dim_customer")

dim_product = silver("dbo.products").select(
    F.xxhash64("product_id").alias("product_key"), "product_id", "product_name",
    "category", "sub_category", "unit_cost", "unit_list_price")
write_gold(dim_product, "dim_product")

StatementMeta(, 2f7172f6-3b4f-4929-b9ad-a3b210f9dc78, 4, Finished, Available, Finished, False)

dim_date 1461
dim_customer 793
dim_product 1862


In [4]:
%%sql
CREATE OR REPLACE TABLE dim_region USING DELTA AS
SELECT xxhash64(country, region) AS region_key, country, region
FROM (SELECT DISTINCT country, region FROM lh_silver.dbo.orders);

StatementMeta(, 2f7172f6-3b4f-4929-b9ad-a3b210f9dc78, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
lines  = silver("dbo.order_lines")
orders = silver("dbo.orders").select("order_id", "order_date", "ship_mode", "customer_id", "country", "region")
ship   = silver("dbo.shipments").select("order_line_id", "ship_date", "qty_shipped")

fact_sales = (lines.join(orders, "order_id")
    .join(ship, lines.row_id == ship.order_line_id, "left")
    .select(
        F.col("row_id").alias("order_line_id"), "order_id",
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("order_date_key"),
        F.date_format("ship_date", "yyyyMMdd").cast("int").alias("ship_date_key"),
        F.xxhash64("customer_id").alias("customer_key"),
        F.xxhash64("product_id").alias("product_key"),
        F.xxhash64("country", "region").alias("region_key"),
        "ship_mode",
        F.col("quantity").alias("qty_ordered"),
        F.coalesce("qty_shipped", F.lit(0)).alias("qty_shipped"),
        "sales", "discount", "profit",
        (F.col("sales") - F.col("profit")).alias("cost"),
        F.datediff("ship_date", "order_date").alias("days_to_ship")))
write_gold(fact_sales, "fact_sales")

region = spark.table("dim_region").select("region", "region_key")
fact_inventory = (silver("dbo.inventory_snapshots").join(region, "region", "left")
    .select(F.date_format("snapshot_date", "yyyyMMdd").cast("int").alias("snapshot_date_key"),
            F.xxhash64("product_id").alias("product_key"), "region_key",
            "qty_on_hand", "qty_on_order"))
write_gold(fact_inventory, "fact_inventory")

StatementMeta(, 2f7172f6-3b4f-4929-b9ad-a3b210f9dc78, 9, Finished, Available, Finished, False)

fact_sales 9994
fact_inventory 1549184


In [8]:
%%sql
OPTIMIZE fact_sales;
OPTIMIZE fact_inventory;

StatementMeta(, 2f7172f6-3b4f-4929-b9ad-a3b210f9dc78, 11, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>